# 06 — Journalisten-Query

Liest [`data/accounts.csv`](../data/accounts.csv) und baut **eine** Brandwatch-Query
für alle Journalist:innen.

**Filter:**
- `category == "News"`
- `label == "Journalist"`
- `channel ∈ {x, instagram, facebook}`

**Struktur:** pro Plattform ein OR-Block (Reihenfolge: X → Instagram → Facebook,
leere Plattformen werden übersprungen). Platzsparend — Handles pro Block auf einer
Zeile, ein `\n` pro Block.

**Output:** `output/queries/journalists_query.txt`.

In [1]:
import os

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR  = os.path.join(PROJECT_ROOT, "output", "queries")
ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
OUTPUT_FILE  = os.path.join(QUERIES_DIR, "journalists_query.txt")

PLATFORM_ORDER = [
    ("x",         "X (Twitter)"),
    ("instagram", "Instagram"),
    ("facebook",  "Facebook"),
]

LANGUAGE_FILTER = "language:de"

os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. Daten laden + filtern

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)
allowed_channels = [p[0] for p in PLATFORM_ORDER]

journos = (
    accounts[
        (accounts["category"] == "News")
        & (accounts["label"] == "Journalist")
        & accounts["channel"].isin(allowed_channels)
    ]
    .dropna(subset=["handle"])
    .drop_duplicates(subset=["channel", "handle"])
    .copy()
)

print(f"Total Journalisten: {len(journos)}")
print()
print(journos["channel"].value_counts())

Total Journalisten: 1832

channel
x            1194
instagram     638
Name: count, dtype: int64


## 2. Helper

In [3]:
def bw_author(handle: str) -> str:
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def block(comment: str, handles) -> str:
    body = " OR ".join(bw_author(h) for h in handles)
    return f"<<< {comment} — {len(handles)} Handles >>>\n({body})"

## 3. Blöcke pro Plattform

In [4]:
blocks: list[str] = []

for channel_key, platform_label in PLATFORM_ORDER:
    sub = journos[journos["channel"] == channel_key]
    if sub.empty:
        print(f"{platform_label:<15s} — keine Handles, übersprungen")
        continue
    handles = (
        sub["handle"]
        .sort_values(key=lambda s: s.str.lower())
        .tolist()
    )
    blocks.append(block(f"Journalist:innen — {platform_label}", handles))
    print(f"{platform_label:<15s} {len(handles):>5} Handles")

print(f"\nBlöcke gesamt: {len(blocks)}")

X (Twitter)      1194 Handles
Instagram         638 Handles
Facebook        — keine Handles, übersprungen

Blöcke gesamt: 2


## 4. Gesamt-Query zusammensetzen

In [5]:
total_handles = sum(b.count('author:"') for b in blocks)
header = f"<<< Journalist:innen — {total_handles} Handles in {len(blocks)} Plattformen >>>"

body = "\nOR\n".join(blocks)
query = f"{header}\n({LANGUAGE_FILTER} AND (\n{body}\n))\n"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(query)

size = os.path.getsize(OUTPUT_FILE)
print(f"Datei:           {OUTPUT_FILE}")
print(f"Größe:           {size:,} bytes  ({size / 1024:.1f} KiB)")
print(f"Handles gesamt:  {total_handles}")
print(f"Blöcke:          {len(blocks)}")

if size > 100_000:
    print(f"\n⚠️  {size - 100_000:,} Zeichen über dem 100k-Limit.")

Datei:           /Users/zorbeyozcan/Projekte/query_printer/output/queries/journalists_query.txt
Größe:           44,561 bytes  (43.5 KiB)
Handles gesamt:  1832
Blöcke:          2


## 5. Preview

In [6]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    content = f.read()

for line in content.split("\n"):
    if "<<<" in line and ">>>" in line:
        print(line)

<<< Journalist:innen — 1832 Handles in 2 Plattformen >>>
<<< Journalist:innen — X (Twitter) — 1194 Handles >>>
<<< Journalist:innen — Instagram — 638 Handles >>>
